# 02 - Spreading Validation

Validate that PDF extraction + fuzzy mapping preserves data integrity vs. original synthetic data.

In [ ]:
import pandas as pd
import numpy as np

# Ground truth
truth = pd.read_csv("../data/synthetic/financials.csv")

# Extracted + mapped
extracted = pd.read_csv("../data/processed/spread_line_items.csv")

print("Ground truth rows:", len(truth))
print("Extracted rows:", len(extracted))
print("Coverage: {:.1%}".format(len(extracted) / len(truth)))

In [ ]:
# Join on borrower_id, fiscal_year, standard_line_item
key = ["borrower_id", "fiscal_year", "standard_line_item"]
merged = truth.merge(
    extracted[key + ["amount"]].rename(columns={"amount": "extracted_amount"}),
    on=key, how="left"
)
merged["diff"] = (merged["amount"] - merged["extracted_amount"]).abs()
merged["match"] = merged["diff"] < 1.0  # tolerance 1 BDT

match_rate = merged["match"].mean()
print(f"Match rate: {match_rate:.2%}")
merged[~merged["match"]].head(20)

In [ ]:
# Confidence score distribution from mapping
import matplotlib.pyplot as plt
extracted["confidence_score"].hist(bins=20, color="darkgreen")
plt.title("Fuzzy Mapping Confidence Score Distribution")
plt.xlabel("Score")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

## Observations

- Match rate target: > 95%
- Low-confidence rows (< 80) are auto-dropped by mapper
- Remaining discrepancies are typically rounding differences